<div style="text-align: center;">

# Postal-code hazard screening (Spain)

</div>

This notebook shows an end-to-end workflow to **sample points inside a postal-code polygon**, retrieve hazard data, and run a lightweight exploratory data analysis (EDA) with interpretable aggregations for an insurance context.

**Workflow:** postal code → points (lat, lon) → hazard request → point-level response → EDA and aggregation by RP (return period), plus an integrated score based on annual exceedance probability and expected annual intensity.


## Objective and scope

- We use a shapefile of Spanish postal codes (CRS: EPSG:4258).
- For a selected postal code, we generate `n` uniformly sampled points inside its polygon.
- We query point-level hazards for several combinations:
  - `event_type x scenario x year` (each combination is one request item).


## Dependencies and helpers

In [1]:
import os

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv
from shapely.geometry import Point
from shapely.prepared import prep

In [2]:
_ = load_dotenv("../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")

## Available hazards

In [3]:
request = {
"include_all": False,
"use_case_id": "ECB_SCORES",
"selected_hazards_list": [],
}
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

# get available hazard data
available_sources = requests.post(
    url=f"{API_BASE}/api/get_available_sources", json=request, headers=headers
)

dict_available_sources = available_sources.json()

In [4]:
def available_sources_to_table(available_sources: dict[str, dict]) -> pd.DataFrame:
    rows = []

    hazards: list[str] = list(available_sources["hazards"].keys())

    for h in hazards:
        h_dict: dict[str, dict] = available_sources["hazards"][h]
        indicators: list[str] = list(h_dict.keys())

        for ind in indicators:
            available_data: dict[str, list | str] = h_dict[ind]

            available_data_scenarios: list[dict] = available_data["scenarios"]
            scenarios_n_years: list[dict] = [{"scenario": d["id"], "years": d["years"]} for d in available_data_scenarios]

            asset_types: list[str] = available_data['available_for_assets_type']

            rows.append(
                {
                    "hazard": h,
                    "indicator": ind,
                    "scenario-year": scenarios_n_years,
                    "asset_types": asset_types
                }
            )

    return pd.DataFrame(rows)

In [5]:
available_sources_to_table(dict_available_sources)

,hazard,indicator,scenario-year,asset_types
0,Subsidence,subsidence_susceptability,"[{'scenario': 'historical', 'years': [1980]}]","[RealEstateAsset, ThermalPowerGeneratingAsset,..."
1,RiverineInundation,flood_depth,"[{'scenario': 'historical', 'years': [1985]}, ...","[RealEstateAsset, PowerInfrastructureAsset, Tr..."
2,CoastalInundation,flood_depth,"[{'scenario': 'historical', 'years': [1971]}, ...","[RealEstateAsset, PowerInfrastructureAsset, Tr..."
3,ChronicHeat,mean_degree_days/above/index,"[{'scenario': 'historical', 'years': [2005]}, ...","[RealEstateAsset, TelecommunicationAsset]"
4,ChronicHeat,mean_degree_days/above/32c,"[{'scenario': 'historical', 'years': [2005]}, ...",[IndustrialActivity]
5,ChronicHeat,mean_work_loss/high,"[{'scenario': 'historical', 'years': [2005]}, ...",[IndustrialActivity]
6,Fire,fire_probability,"[{'scenario': 'historical', 'years': [2010]}, ...","[RealEstateAsset, PowerInfrastructureAsset, Tr..."
7,Drought,spi6,"[{'scenario': 'historical', 'years': [1995]}, ...","[RealEstateAsset, ThermalPowerGeneratingAsset,..."
8,Drought,cdd,"[{'scenario': 'historical', 'years': [1995]}, ...","[RealEstateAsset, ThermalPowerGeneratingAsset,..."
9,Wind,max_speed,"[{'scenario': 'historical', 'years': [2010]}, ...","[RealEstateAsset, IndustrialActivity, PowerInf..."


## Load postal codes

In [6]:
gdf = gpd.read_file("../resources/codigos_postales/codigos_postales.shp")

# Identify ZIP code columns
print("Postal Codes columns: ", list(gdf.columns))
print("Geometry CRS: ", gdf.crs)

Postal Codes columns:  ['ID_CP', 'COD_POSTAL', 'ALTA_DB', 'CODIGO_INE', 'geometry']
Geometry CRS:  EPSG:4258


## 1) Postal-code selection and point sampling

We select an example postal code (Seville, near the Guadalquivir River) and generate `n` uniformly sampled points inside its polygon.

### Why we project to meters before sampling
The shapefile uses EPSG:4258 (degrees). To make sampling spatially consistent, we temporarily project the geometry to a metric CRS (EPSG:25830), where:
- the bounding box and sampling are distance-consistent,
- areas and margins, if used, have physical units.

After sampling, we return the points in EPSG:4326 (standard latitude/longitude) to avoid conflicts with the hazard API.


In [7]:
def sample_points_per_cp(
    gdf_cp: gpd.GeoDataFrame,
    cp: str | int,
    *,
    postcode_col: str = "COD_POSTAL",
    n: int = 50,
    source_crs: str = "EPSG:4258",     # Spanish shapefile CRS (ETRS89)
    sampling_crs: str = "EPSG:25830",  # meters (ETRS89 / UTM 30N)
    out_crs: str = "EPSG:4326",        # standard output CRS for the library
    seed: int = 42,
    margin_m: float = 0.0,
    batch_mult: int = 10,
    batch_hard_cap: int = 50_000,
    max_iter_factor: int = 60,
) -> pd.DataFrame:
    """
    Given a postcode, finds its polygon in gdf_cp and samples n points inside it.
    Returns a DataFrame with lat/lon in EPSG:4326 (WGS84) for building hazard requests.
    """

    # Checks
    if gdf_cp.crs is None:
        raise ValueError("GeoDataFrame has no CRS.")
    if str(gdf_cp.crs).upper() != source_crs.upper():
        raise ValueError(f"Unexpected CRS: {gdf_cp.crs}. Expected {source_crs}.")

    if postcode_col not in gdf_cp.columns:
        raise ValueError(f"Column not found: '{postcode_col}'. Columns: {list(gdf_cp.columns)}")

    # Normalize postcode (accepts int or str; preserves leading zeros)
    cp_str = str(cp).strip()
    if cp_str.isdigit() and len(cp_str) < 5:
        cp_str = cp_str.zfill(5)

    # Compare against the normalized column as a string without mutating the original gdf
    col = gdf_cp[postcode_col].astype(str).str.strip()
    col = col.apply(lambda s: s.zfill(5) if s.isdigit() and len(s) < 5 else s)

    match = gdf_cp.loc[col == cp_str, "geometry"]
    if match.empty:
        raise ValueError(f"Postal Code {cp_str} not found in column {postcode_col}.")

    geom = match.iloc[0]
    if geom is None or geom.is_empty:
        raise ValueError(f"Empty geometry for Postal Code {cp_str}.")

    # Light repair if needed
    if not geom.is_valid:
        geom = geom.buffer(0)

    # Project to a metric CRS for sampling
    geom_m = gpd.GeoSeries([geom], crs=source_crs).to_crs(sampling_crs).iloc[0]
    geom_prep = prep(geom_m)

    minx, miny, maxx, maxy = geom_m.bounds
    minx -= margin_m
    miny -= margin_m
    maxx += margin_m
    maxy += margin_m

    rng = np.random.default_rng(seed)
    batch_size = min(max(n * batch_mult, 1000), batch_hard_cap)
    max_iter = max(n * max_iter_factor, 2000)

    pts_m = []
    it = 0
    while len(pts_m) < n and it < max_iter:
        xs = rng.uniform(minx, maxx, size=batch_size)
        ys = rng.uniform(miny, maxy, size=batch_size)
        cand = [Point(x, y) for x, y in zip(xs, ys)]
        pts_m.extend([p for p in cand if geom_prep.contains(p)])
        it += batch_size

    if len(pts_m) < n:
        raise RuntimeError(
            f"Unable to sample {n} points in Postal Code {cp_str} (only {len(pts_m)} were sampled).",
            "Try increasing batch_mult/max_iter_factor or revisit the geometry."
        )

    pts_m = pts_m[:n]

    # Convert to EPSG:4326 before returning lat/lon
    pts_4326 = gpd.GeoSeries(pts_m, crs=sampling_crs).to_crs(out_crs)

    return pd.DataFrame({
        "lat": pts_4326.y,
        "lon": pts_4326.x,
    })

In [8]:
cp = "41010"

points = sample_points_per_cp(gdf, cp, n=50)

## 2) Build the hazard request

The request is built as a list of `items`. Each item is one combination of:

- `event_type` (hazard), for example `RiverineInundation` or `Wind`
- `scenario`, for example `rcp4p5`, `rcp8p5`, or `historical`
- `year`, for example 2035, 2085, or 1971
- `latitudes` and `longitudes`, in the same order as the `points` DataFrame

Each item is identified by `request_item_id`, which is useful for reconciling outputs.


In [9]:
lons = points["lon"].tolist()
lats = points["lat"].tolist()

items = []

# Flood 
for scenario, year in [("rcp4p5", 2035), ("rcp4p5", 2085), ("rcp8p5", 2035), ("rcp8p5", 2085), ("historical", 1971)]:
    items.append({
        "request_item_id": f"riverine_{scenario}_{year}",
        "event_type": "RiverineInundation",
        "longitudes": lons,
        "latitudes": lats,
        "year": year,
        "scenario": scenario,
        "indicator_id": "flood_depth",
    })

# Wind 
items.append({
        "request_item_id": "wind_historical_1999",
        "event_type": "Wind",
        "longitudes": lons,
        "latitudes": lats,
        "year": 1999,
        "scenario": "historical",
        "indicator_id": "wind_speed/3s",
})

request_dict = {"items": items, "interpolation": "floor"}

## 3) Response structure

The API returns one output per item (one `event_type x scenario x year` combination).

Each item contains:

- `request_item_id`, `event_type`, `scenario`, `year`
- `intensity_curve_set`: a list of curves, **one per point** (with the same length as the input)

Each curve contains:
- `index_values`: a list of return periods, for example 10, 30, 100, 300, and 1000
- `intensities`: intensity values for those RPs

Because the curves are ordered in the same way as the submitted points, they can be reconciled directly with `points`.


In [10]:
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

# get available hazard data
hazard_data = requests.post(
    url=f"{API_BASE}/api/get_hazard_data", json=request_dict, headers=headers
)

dict_hazard_data = hazard_data.json()

## 4) Parse into wide tables (`rp10`, `rp30`, `rp100`, ...)

For EDA, it is convenient to build one DataFrame per item with these columns:

- `lat`, `lon`
- `rp10`, `rp30`, `rp100`, `rp300`, `rp1000` (depending on the values returned in `index_values`)
- metadata: `request_item_id`, `event_type`, `scenario`, `year`

This enables us to:
- map `rp100`,
- calculate percentiles by RP,
- compare scenarios and years.


In [11]:
def _rps_del_item(item: dict) -> list[float]:
    """Extract RPs (index_values) using first curve that contains index_values."""
    for c in item.get("intensity_curve_set", []):
        rps = c.get("index_values", []) or []
        if rps:
            return [float(rp) for rp in rps]
    return []

def build_rp_tables_por_item(
    items: list[dict],
    points: pd.DataFrame,
) -> dict[tuple[str, str, int], pd.DataFrame]:
    """
    Returns dict: (event_type, scenario, year) -> DataFrame with lat, lon, rpXX...
    """
    base = points.reset_index(drop=True).copy()
    out = {}

    for item in items:
        curves = item["intensity_curve_set"]  # one curve per point (same order than points)
        event_type = item.get("hazard_type")
        scenario = item.get("scenario")
        year = int(item.get("year"))
        rid = item.get("request_item_id")
        indicator = item.get("indicator_id") or item.get("model")

        if len(curves) != len(base):
            raise ValueError(f"{rid}: {len(curves)} curves were received but {len(base)} points were expected.")

        rps = _rps_del_item(item)
        if not rps:
            raise ValueError(f"{rid}: index_values not found in any curve.")

        # Matrix (n_points x n_rps)
        mat = []
        for c in curves:
            rp_map = {
                float(rp): float(iv)
                for rp, iv in zip(c.get("index_values", []) or [], c.get("intensities", []) or [])
            }
            mat.append([rp_map.get(rp, np.nan) for rp in rps])

        arr = np.asarray(mat, float)

        df = base.copy()
        df["request_item_id"] = rid
        df["event_type"] = event_type
        df["indicator"] = indicator
        df["scenario"] = scenario
        df["year"] = year

        for j, rp in enumerate(rps):
            col = f"rp{int(rp)}" if float(rp).is_integer() else f"rp{str(rp).replace('.', '_')}"
            df[col] = arr[:, j]

        key = (event_type, scenario, year)
        if key in out:
            raise ValueError(f"Duplicated key: {key}. Revisit uniqueness event_type×scenario×year.")
        out[key] = df

    return out

In [12]:
rp_tables = build_rp_tables_por_item(dict_hazard_data["items"], points)

df_riverine_2035 = rp_tables[("RiverineInundation", "rcp4p5", 2035)]

In [13]:
rps = ["rp10", "rp30", "rp100", "rp300", "rp1000"]

# Basic summary
df_riverine_2035[rps].describe()

,rp10,rp30,rp100,rp300,rp1000
count,33.000000,37.000000,47.000000,40.000000,42.000000
mean,4.678988,4.710062,4.108138,5.132551,5.181657
std,2.985330,3.149857,3.505255,3.316296,3.423629
min,0.869600,0.057680,0.000000,0.075120,0.242900
25%,1.990000,2.405000,0.993750,2.715250,2.764750
50%,4.259000,4.356000,3.247000,4.643000,4.221500
75%,6.432000,6.844000,5.872000,7.648000,7.596250
max,10.680000,11.170000,11.660000,11.990000,12.300000


In [14]:
# NaNs per RP
df_riverine_2035[rps].isna().sum()

rp10      17
rp30      13
rp100      3
rp300     10
rp1000     8
dtype: int64

In [15]:
# Zeros ratio (Warning: it may be *no inundation* or *no data* depending on the dataset)
(df_riverine_2035[rps] == 0).mean()

rp10      0.00
rp30      0.00
rp100     0.16
rp300     0.00
rp1000    0.00
dtype: float64

In [16]:
coverage = (1 - df_riverine_2035[rps].isna().mean()) * 100 # %
coverage

rp10      66.0
rp30      74.0
rp100     94.0
rp300     80.0
rp1000    84.0
dtype: float64

## 5) EDA: point map colored by intensity

First visual check: map a specific RP, such as `rp100`, to inspect spatial heterogeneity within the postal code.

- Color = intensity at `rp100`
- Hover = (lat, lon, rp100) and/or additional metrics



In [17]:
def plot_map_points(df: pd.DataFrame, value_col: str = "rp100", title: str | None = None):
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        color=value_col,
        hover_data=["lat", "lon", value_col],
        zoom=12,
        height=650,
    )
    center = {"lat": float(df["lat"].mean()), "lon": float(df["lon"].mean())}
    fig.update_layout(
        map_style="open-street-map",
        map_center=center,
        title=title or f"{value_col} per point",
        margin=dict(l=0, r=0, t=50, b=0),
    )
    return fig

In [18]:
fig = plot_map_points(df_riverine_2035, "rp100", "Riverine inundation (2035, rcp4p5) – rp100")
fig.show()

## 6) EDA: aggregated curve by RP (median and p10-p90 band)

Second plot: aggregate all points in the postal code for each return period:

- median (p50) as the central curve
- p10-p90 band to show spatial dispersion

We use a logarithmic X-axis because return periods are commonly compared on a log scale.


In [19]:
def plot_agg_curve(df: pd.DataFrame, rp_list=(10, 30, 100, 300, 1000), title: str | None = None):
    rps = np.array(rp_list, dtype=float)
    cols = [f"rp{int(rp)}" for rp in rps]

    vals = df[cols].to_numpy(float)

    p10 = np.nanpercentile(vals, 10, axis=0)
    p50 = np.nanpercentile(vals, 50, axis=0)
    p90 = np.nanpercentile(vals, 90, axis=0)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=rps, y=p90, mode="lines", line=dict(width=0), showlegend=False, name = "p90"))
    fig.add_trace(go.Scatter(x=rps, y=p10, mode="lines", line=dict(width=0),
                             fill="tonexty", name="p10"))
    fig.add_trace(go.Scatter(x=rps, y=p50, mode="lines+markers", name="mediana"))

    fig.update_layout(
        title=title or "Aggregated curve (median and p10–p90 band)",
        xaxis_title="Return period [years]",
        yaxis_title="Intensity - Flood Depth [m]",
        xaxis_type="log",
        height=450,
    )
    return fig

In [20]:
fig = plot_agg_curve(df_riverine_2035, title="Riverine Inundation (2035, rcp4p5) – aggregated curve")
fig.show()

## 7) RP-level aggregation table

For each RP, we calculate point-level summary metrics:

- `mean`, `p50`, `p95`, and `max` (ignoring NaNs)
- `share_gt0`: fraction of points with intensity > 0, useful as a proxy for affected area
- `coverage`: fraction of points with available data (not NaN)


In [21]:
def aggregate_rp_stats(df: pd.DataFrame, rp_cols: list[str]) -> pd.DataFrame:
    """
    Return a table per RP with:
      - mean, p50, p95, max (ignoring NaNs)
      - share_gt0 (points ratio with intensity > 0)
      - coverage (points ratio with no-NaN data)
      - n (number of data points)
    """
    rows = []
    n_total = len(df)

    for c in rp_cols:
        s = df[c].astype(float)
        mask = ~np.isnan(s.to_numpy())

        if mask.sum() == 0:
            rows.append({
                "rp": c,
                "mean": np.nan, "p50": np.nan, "p95": np.nan, "max": np.nan,
                "share_gt0": np.nan,
                "coverage": 0.0,
                "n": 0,
            })
            continue

        s_valid = s[mask]

        rows.append({
            "rp": c,
            "mean": float(np.nanmean(s_valid)),
            "p50": float(np.nanpercentile(s_valid, 50)),
            "p95": float(np.nanpercentile(s_valid, 95)),
            "max": float(np.nanmax(s_valid)),
            "share_gt0": float(np.mean(s_valid > 0.0)),
            "coverage": float(mask.sum() / n_total),
            "n": int(mask.sum()),
        })

    return pd.DataFrame(rows)

In [22]:
# Automatically detects the rp columns in case the list changes
rp_cols = sorted([c for c in df_riverine_2035.columns if c.startswith("rp")], key=lambda x: float(x[2:].replace("_",".")))

rp_stats = aggregate_rp_stats(df_riverine_2035, rp_cols)
rp_stats

,rp,mean,p50,p95,max,share_gt0,coverage,n
0,rp10,4.678988,4.2590,9.9310,10.68,1.000000,0.66,33
1,rp30,4.710062,4.3560,10.4400,11.17,1.000000,0.74,37
2,rp100,4.108138,3.2470,10.8610,11.66,0.829787,0.94,47
3,rp300,5.132550,4.6430,11.2400,11.99,1.000000,0.80,40
4,rp1000,5.181657,4.2215,11.5275,12.30,1.000000,0.84,42


## 8) Hazard-level aggregation with exceedance probability (EAI)

We want a scalar value per point, and then per postal code, that summarizes hazard intensity and is easy to aggregate.

### Annual exceedance probability (AEP)
For a return period `RP` in years, annual exceedance probability is approximated as:

$$
p = \frac{1}{RP}
$$

### Expected Annual Intensity (EAI) as an integral
We convert the curve $$I(RP)$$ into $$I(p)$$ and calculate the area under the curve:

$$
\mathrm{EAI} \approx \int I(p)\, dp
$$

Because we only have discrete values (a small number of RPs), we use the trapezoidal rule:

$$
\mathrm{EAI} \approx \sum_{i=1}^{k-1} \frac{I_i + I_{i+1}}{2}\,(p_{i+1}-p_i)
$$

Notes:
- **0.0 is treated as 0.0** (no inundation) and reduces the area.
- **NaN is ignored** (missing data).
- At least two curve points are required for integration.

EAI is a **proxy for annual hazard severity**.


In [23]:
def eai_from_row(row: pd.Series, rp_cols: list[str]) -> float:
    """
    Expected Annual Intensity (proxy): integrate I(p) dp with p=1/RP.
    Ignore NaNs. Keep 0.0 as 0.0.
    """
    pairs = []
    for c in rp_cols:
        v = row.get(c, np.nan)
        if pd.notna(v):
            rp = float(c[2:].replace("_", "."))
            pairs.append((rp, float(v)))

    if len(pairs) < 2:
        return np.nan

    rps = np.array([p[0] for p in pairs], dtype=float)
    intens = np.array([p[1] for p in pairs], dtype=float)

    p = 1.0 / rps
    order = np.argsort(p)  # increasing p
    p = p[order]
    intens = intens[order]

    return float(np.trapezoid(intens, p))

def aggregate_eai(df: pd.DataFrame, rp_cols: list[str]) -> dict:
    eai = df.apply(eai_from_row, axis=1, rp_cols=rp_cols)
    out = {
        "eai_mean": float(np.nanmean(eai)),
        "eai_p50": float(np.nanpercentile(eai, 50)),
        "eai_p95": float(np.nanpercentile(eai, 95)),
        "eai_max": float(np.nanmax(eai)),
        "eai_coverage": float(np.mean(~np.isnan(eai))),
    }
    return out

In [24]:
df_riverine_2035 = df_riverine_2035.copy()
df_riverine_2035["eai"] = df_riverine_2035.apply(eai_from_row, axis=1, rp_cols=rp_cols)

eai_summary = aggregate_eai(df_riverine_2035, rp_cols)
print(eai_summary)

{'eai_mean': 0.42384338475000016, 'eai_p50': 0.37510175, 'eai_p95': 1.0288266666666668, 'eai_max': 1.101855, 'eai_coverage': 0.8}


## 9) Visualization

To make the result easier to interpret, we add distribution plots:

- histogram + boxplot of `rp100`
- histogram + boxplot of `EAI`

This helps explain:
- heterogeneity within the postal code,
- the proportion of points with no risk signal (0.0),
- distribution shape and outliers.



In [25]:
def plot_distribution_rp(df, rp_col="rp100", title=None):
    s = df[rp_col].astype(float)

    df_plot = df.loc[~s.isna(), [rp_col]].copy()
    fig = px.histogram(
        df_plot,
        x=rp_col,
        nbins=25,
        marginal="box", 
        title=title or f"{rp_col} distribution (NaN excluded, 0.0 included)",
    )
    return fig

In [26]:
fig = plot_distribution_rp(df_riverine_2035, "rp100", "Riverine Inundation (2035, rcp4p5) – rp100 distribution")
fig.show()

In [27]:
fig = px.histogram(
    df_riverine_2035.loc[~df_riverine_2035["eai"].isna(), :],
    x="eai",
    nbins=25,
    marginal="box",
    title="EAI distribution (AEP-integrated) along the Postal Code",
)
fig.show()